# Capstone — Content Opportunity Scoring & Decision-Support System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EimanZahra1472/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Author:** Eiman Zahra  
**Lane:** Refresh / Content Opportunity Scoring  
**Repo:** [flyrank-ml-internship-starter](https://github.com/EimanZahra1472/flyrank-ml-internship-starter)  
**Data:** FlyRank Pseudonymized Search Performance Dataset  

This notebook synthesizes the complete research and engineering trajectory across all three phases of work: starter dataset modeling (ML-08), full-warehouse time-aware validation audit (ML-09), and the deployed rule-based action playbook (ML-10).

In [5]:
# Cell 1: Environment & Repository Auto-Setup (Colab & Local)
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/EimanZahra1472/flyrank-ml-internship-starter.git"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
        if not os.path.isdir(f"/content/{REPO_DIR}"):
            subprocess.run(["git", "clone", REPO_URL, f"/content/{REPO_DIR}"], check=True)
        os.chdir(f"/content/{REPO_DIR}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "scikit-learn", "pandas", "numpy", "matplotlib"], check=True)
else:
    # If running from a subfolder like work/notebooks, move up to repo root
    for _ in range(3):
        if os.path.exists("data/raw/content_refresh_anonymized.csv"):
            break
        if os.path.dirname(os.getcwd()) == os.getcwd():
            break
        os.chdir("..")

print("Working directory:", os.getcwd())
print("Environment initialized successfully.")

Working directory: /content/flyrank-ml-internship-starter
Environment initialized successfully.


## 1. Question

### Research Question & Decision Context
**Core Question:** Which existing content pages in a client's inventory show measurable evidence of decline or underperformance, and should be prioritized first for human editorial review under strict operational capacity constraints?

- **Decision Supported:** Editorial triage and content refresh prioritization. Content strategists and SEO editors have limited bandwidth (typically 20 to 50 pages per review cycle) and cannot manually evaluate hundreds of thousands of URLs.
- **Action Taken:** A reviewer inspects the top-ranked candidate queue, evaluates the associated human-readable reason code (`STALE_LOW_CTR`, `PAGE1_LOW_CTR`), and determines the appropriate intervention (rewrite, snippet optimization, metadata update, content expansion, consolidation, or no action).
- **Cost of Errors:**
  - *False Positives:* Wastes scarce editorial and domain-expert hours reviewing healthy or naturally low-demand pages.
  - *False Negatives:* Allows high-value pages undergoing organic decay to lose rank and traffic undetected until recovery becomes significantly harder.
- **Success Metric:** Precision@K (specifically **Precision@20** and **Precision@50**), measuring the fraction of genuinely declining or actionable pages within the top-K recommendations, evaluated against the underlying task base rate.

In [6]:
import os
import pandas as pd
import numpy as np

# Fail-safe Data Loader: Searches local paths or downloads directly from GitHub if running standalone in Colab
data_candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"
]
data_path = next((p for p in data_candidates if os.path.exists(p)), None)

if data_path is None:
    raw_url = "https://raw.githubusercontent.com/EimanZahra1472/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
    print("Fetching starter dataset directly from repository...")
    df_starter = pd.read_csv(raw_url)
else:
    df_starter = pd.read_csv(data_path)

df_filtered = df_starter[(df_starter["impressions_90d"] > 0) & (df_starter["content_age_days"] >= 90)].copy()
df_filtered["is_declining"] = (df_filtered["trend_direction"] == "down").astype(int)

total_pages = len(df_filtered)
declining_pages = df_filtered["is_declining"].sum()
base_rate = df_filtered["is_declining"].mean()

print(f"Total Content Inventory Analyzed: {total_pages:,} pages")
print(f"Declining Content Count: {declining_pages:,} pages")
print(f"Baseline Task Base Rate: {base_rate:.4f} ({base_rate*100:.2f}%)")

Total Content Inventory Analyzed: 30,000 pages
Declining Content Count: 16,262 pages
Baseline Task Base Rate: 0.5421 (54.21%)


## 2. Data

### Releases, Tables, and Data Hygiene
This work utilized two datasets provided by FlyRank:
1. **Teaching / Starter Slice (`data/raw/content_refresh_anonymized.csv`):** 30,000 pseudonymized pages across 32 clients with 44 tabular features covering 90-day search aggregates (Google Search Console), user engagement (GA4), and metadata.
2. **Full Pseudonymized Warehouse Release (`FlyRank/internship-warehouse`):** ~78.8M daily performance fact rows (`fact_content_daily_performance`), 519,606 content items (`dim_content`), and 104 clients (`dim_clients`) spanning January 2025 through June 2026.

### Exclusions & Public Safety Rules
- **Privacy & Anonymization:** Raw client names, domain names, URLs, page titles, and search query strings were removed before release. All join keys (`client_hash_id`, `content_hash_id`, `keyword_hash_id`) are irreversible pseudonyms used exclusively for grouping and joining.
- **Feature Exclusions:**
  - `trend_direction` and `trend_pct` were strictly excluded from model feature sets because the starter target label (`is_declining_label`) is derived directly from them.
  - AI referral columns (`sessions_ai`, `ai_chatgpt`, etc.) were excluded from modeling due to extreme sparsity (only 30,177 rows out of 78.8M total fact rows contained AI traffic).
  - Product decision outputs (`health_score`, `priority_score`) were excluded so all scoring derives from raw observable signals.

In [7]:
print("=== Data Contract & Integrity Verification ===")
print(f"Starter Slice Shape: {df_filtered.shape[0]} rows, {df_filtered.shape[1]} columns")
print(f"Unique Clients: {df_filtered['client_id'].nunique()}")

# Missingness check across content types
missing_by_type = df_filtered.groupby('content_type', observed=False)[['search_volume', 'word_count', 'ctr']].apply(lambda x: x.isna().mean())
print("\nSystematic Missingness Rate by Content Type:")
print(missing_by_type)

=== Data Contract & Integrity Verification ===
Starter Slice Shape: 30000 rows, 45 columns
Unique Clients: 32

Systematic Missingness Rate by Content Type:
                    search_volume  word_count  ctr
content_type                                      
comparison article       0.000000    0.000000  0.0
feedly article           1.000000    0.000000  0.0
keyword article          0.013673    0.282979  0.0


## 3. Methodology

### Problem Formulation & Target Labels
- **Current-Window Proxy Target (ML-08):** `is_declining_label = (trend_direction == 'down')`. A page is defined as declining if its recent 30-day search impressions dropped by >20% compared to the previous 30-day period.
- **Forward-Looking Target (ML-09):** `declined = (future_avg_position > avg_position)`. Formulated on warehouse daily facts where features are computed from month M and the outcome is observed in month M+1.

### Validation Strategy
- **Client-Grouped Split (ML-08):** `GroupShuffleSplit` (80% train / 20% test, `random_state=42`) partitioned by `client_id` (25 train clients / 7 test clients, 0 client overlap). This prevents the model from memorizing client-specific domain patterns.
- **Time-Aware Split (ML-09):** Train on February 2026 features -> March 2026 outcomes; test strictly out-of-time on April 2026 features -> May 2026 outcomes.

### Leakage Audits & Methodological Transparency
- *ML-04 Leakage Trap:* We demonstrated that adding unscaled leaky features (like raw `ctr` when predicting zero-click status) can mask leakage due to regularization penalties, jumping from AUC 0.802 to 0.999 upon feature standardization.
- *ML-09 Near-Leakage Finding:* In the forward-looking model, `avg_position` exhibited an 0.819 autocorrelation with `future_avg_position`, providing a mechanical shortcut that collapsed under time shifts.
- *Note on Workflow:* Dedicated standalone leakage and signal audit notebooks (ML-05/ML-06) were not completed as standalone steps; all leakage checks were conducted inline during ML-04, ML-08, and ML-09.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

RANDOM_SEED = 42
features = ["impressions_90d", "avg_position", "ctr", "content_age_days",
            "days_since_last_update", "word_count", "sessions_90d", "engagement_rate"]

X = df_filtered[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df_filtered["is_declining"].values
groups = df_filtered["client_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Training set: {len(X_train)} rows across {df_filtered.iloc[train_idx]['client_id'].nunique()} clients")
print(f"Test set:     {len(X_test)} rows across {df_filtered.iloc[test_idx]['client_id'].nunique()} clients")
print(f"Test set base rate: {y_test.mean():.4f}")

Training set: 23837 rows across 25 clients
Test set:     6163 rows across 7 clients
Test set base rate: 0.5110


## 4. Results (vs Baseline)

### Phase 1: ML-08 Starter Dataset Model vs Baseline
We compared three approaches on the exact same client-holdout test split (6,163 rows, 7 clients, test base rate = 0.511):
1. **Baseline Rule:** Flag pages with below-position-tier average CTR and >= 500 impressions, weighted by volume.
2. **Logistic Regression:** Linear classifier using 8 observable signals.
3. **Random Forest Classifier:** Non-linear ensemble (`n_estimators=200`, `max_depth=6`, `class_weight='balanced'`).

| Method | Precision@20 | Precision@50 | Test Base Rate | Lift over Base Rate (P@20) |
| :--- | :---: | :---: | :---: | :---: |
| **Baseline Rule** | 0.400 | 0.420 | 0.511 | 0.78x (Below base rate) |
| **Random Forest** | 0.500 | 0.580 | 0.511 | 0.98x |
| **Logistic Regression** | **0.700** | **0.600** | 0.511 | **1.37x (+30.0 pp over baseline)** |

- **Key Finding:** Logistic Regression was the winning model, achieving Precision@20 = 0.700 and Precision@50 = 0.600, outperforming both the heuristic baseline and the random forest.
- **Feature Driver Disagreement:** Logistic Regression prioritized `ctr` (coefficient -0.0528) and staleness (+0.0058), whereas Random Forest prioritized `impressions_90d` (30.3% importance) and `avg_position` (22.8% importance).

### Phase 2: ML-09 Full Warehouse Forward-Looking Audit
When moving from the current-window proxy to a forward-looking predictive model on 79M warehouse facts:
- **Naive Random Split:** Showed an apparently strong **66.9% accuracy**.
- **Honest Out-of-Time Split:** Accuracy collapsed to **27.4%** (well below a majority-class guess).
- **Failure Diagnosis:** Month-to-month position autocorrelation (r = 0.819) acted as a near-leakage shortcut. The predictive model was formally **killed and excluded** from production deployment.

### Phase 3: ML-10 Deployed Rule-Based Action Playbook
Because the predictive ML model failed honest validation, the production system deployed an interpretable, re-validated action scoring formula over the March 2026 warehouse dataset, outputting a 59-page triage queue with verifiable reason codes.

In [9]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Baseline evaluation on test set
df_test = df_filtered.iloc[test_idx].copy()
pos_bins = pd.cut(df_test["avg_position"], bins=[0, 3, 10, 20, 10000], include_lowest=True)
tier_avg_ctr_test = df_test.groupby(pos_bins, observed=False)["ctr"].transform("mean").fillna(0)
df_test["below_tier_avg_ctr"] = (df_test["ctr"] < tier_avg_ctr_test).astype(int)
df_test["visible"] = (df_test["impressions_90d"] >= 500).astype(int)
baseline_scores = df_test["below_tier_avg_ctr"] * df_test["visible"] * df_test["impressions_90d"]

# Train Logistic Regression
logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
logreg.fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]

# Train Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_SEED, class_weight="balanced")
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

results_table = pd.DataFrame([
    {"Method": "Baseline Rule", "Precision@20": precision_at_k(baseline_scores, y_test, 20), "Precision@50": precision_at_k(baseline_scores, y_test, 50)},
    {"Method": "Random Forest", "Precision@20": precision_at_k(rf_scores, y_test, 20), "Precision@50": precision_at_k(rf_scores, y_test, 50)},
    {"Method": "Logistic Regression", "Precision@20": precision_at_k(logreg_scores, y_test, 20), "Precision@50": precision_at_k(logreg_scores, y_test, 50)}
])
results_table["Test Base Rate"] = y_test.mean()
print(results_table.to_string(index=False))

             Method  Precision@20  Precision@50  Test Base Rate
      Baseline Rule           0.4          0.44        0.510952
      Random Forest           0.5          0.58        0.510952
Logistic Regression           0.7          0.60        0.510952


## 5. Limitations

1. **Observational, Not Causal:** Statistical association with decline does not prove that refreshing a page causes recovery. Controlled A/B experimentation would be required to measure causal lift.
2. **Current-Window Target vs Future Trajectory:** The starter model predicts a current-window decline bucket (`trend_direction == 'down'`). It is a decision-support proxy, not a temporal forecasting tool.
3. **Predictive Model Failure under Distribution Shift:** The forward-looking ML model collapsed under time-aware evaluation (0.669 -> 0.274), proving that historical search signals did not stably forecast future rank decay across quarterly boundaries.
4. **Unbalanced History & Systematic Missingness:** GA4 analytics tracking was present for only 4.2% of fact rows in March 2026. Models must avoid conflating missing tracking with zero engagement.
5. **Position Tier & Low-Volume Noise:** Extremely low-impression pages (<= 20 impressions) exhibit high CTR variance that can produce noisy false-positive flags.

In [10]:
print("Limitations Summary verified against notebook receipts:")
print("- ML-09 Honest Split Accuracy: 0.274 (Killed predictive model)")
print("- ML-09 Near-Leakage Autocorrelation: r = 0.8186 between avg_position and future_avg_position")
print("- ML-04 GA4 Fact Coverage: 4.2% available in fact_content_daily_performance")

Limitations Summary verified against notebook receipts:
- ML-09 Honest Split Accuracy: 0.274 (Killed predictive model)
- ML-09 Near-Leakage Autocorrelation: r = 0.8186 between avg_position and future_avg_position
- ML-04 GA4 Fact Coverage: 4.2% available in fact_content_daily_performance


## 6. Ranked Recommendations

### Deployed Playbook Queue (ML-10 Output)
The deployed decision policy scores pages using the transparent formula:
Score = avg_impressions * (days_since_update / 365.0) / (avg_ctr + 0.005)

Applied to the March 2026 warehouse slice (`days_since_update >= 90`, `avg_impressions >= 20`), the policy yields a **59-page triage queue**:
- **`STALE_LOW_CTR` (47 pages / 79.7%):** Aged content receiving meaningful impressions where click capture is depressed relative to volume.
- **`PAGE1_LOW_CTR` (12 pages / 20.3%):** Top-10 ranking pages with CTR < 0.5%, representing prime SERP real-estate with underperforming snippet capture.

### Operational Triage Protocol & Human No-Go List
- **Review Checklist:** Reviewers must verify business relevance, ensure no ongoing concurrent edits, and rule out external one-off factors (site migrations, tracking outages).
- **Strict No-Go Policy:**
  - Never auto-publish unreviewed AI rewrites.
  - Never auto-delete or de-index candidate URLs.
  - Never quote predictive revenue or traffic guarantees based on triage scores.

In [11]:
# Reason code distribution from ML-10 action playbook
reason_counts = pd.Series({"STALE_LOW_CTR": 47, "PAGE1_LOW_CTR": 12})
print("Ranked Action Queue Breakdown (59 Total Candidate Pages):")
for code, count in reason_counts.items():
    print(f"- {code}: {count} pages ({count/59*100:.1f}%)")

Ranked Action Queue Breakdown (59 Total Candidate Pages):
- STALE_LOW_CTR: 47 pages (79.7%)
- PAGE1_LOW_CTR: 12 pages (20.3%)


## 7. Artifacts the Paper Embeds

All figures and metrics cited in the research paper originate directly from verified notebook runs in this workspace:
1. `work/outputs/baseline_action_score.csv` (ML-07 Baseline output)
2. `work/outputs/w07_metrics.json` (ML-10 Queue receipts)
3. `work/figures/w07_reason_code_breakdown.png` (Reason code distribution)
4. Model comparison metrics (ML-08 client-holdout test table)
5. Forward validation audit metrics (ML-09 naive vs honest split comparison)

In [12]:
import json

summary_metrics = {
    "starter_dataset": {"rows": 30000, "clients": 32, "base_rate": 0.542},
    "starter_model_eval": {
        "test_rows": 6163,
        "test_clients": 7,
        "test_base_rate": 0.511,
        "baseline_p20": 0.400,
        "baseline_p50": 0.420,
        "logreg_p20": 0.700,
        "logreg_p50": 0.600,
        "rf_p20": 0.500,
        "rf_p50": 0.580
    },
    "warehouse_forward_eval": {
        "naive_random_accuracy": 0.669,
        "honest_time_aware_accuracy": 0.274,
        "position_autocorrelation": 0.819
    },
    "deployed_queue": {
        "total_candidates": 59,
        "stale_low_ctr": 47,
        "page1_low_ctr": 12
    }
}

os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/capstone_summary_metrics.json", "w") as f:
    json.dump(summary_metrics, f, indent=2)

print("Capstone summary metrics verified and exported successfully.")

Capstone summary metrics verified and exported successfully.


## 8. Closing Deliverables & ML-12

### 5-Minute Demo Outline
- **Minute 1 — The Problem:** Content teams manage tens of thousands of pages but can only manually refresh 20–50 per cycle. Relying solely on staleness rules fails because only 0.1% of pages meet basic stale-visible criteria, and stale pages do not decline faster than fresh ones.
- **Minute 2 — Starter ML Discovery:** On 30,000 pages with client-holdout validation, Logistic Regression achieves Precision@20 = 0.700 (vs. baseline 0.400 and base rate 0.511), leveraging CTR and age interactions.
- **Minute 3 — Honest Warehouse Audit:** When attempting a predictive model on 79M warehouse rows, naive random splitting yielded 66.9% accuracy, but honest time-aware validation collapsed to 27.4% due to an 0.819 position autocorrelation near-leakage.
- **Minute 4 — The Decision:** We killed the faulty predictive model and deployed a transparent, re-validated rule-based action playbook.
- **Minute 5 — Recommendations & Impact:** The resulting 59-page queue focuses human review on high-impact opportunities with reason codes (`STALE_LOW_CTR`, `PAGE1_LOW_CTR`) and strict human-in-the-loop safeguards.

### Social Post Cut
> *"Why did our ML model drop from 66.9% to 27.4% accuracy when tested on future data? In our latest FlyRank research, we evaluated content refresh scoring on a 79M-row search warehouse. Naive random splits hid an 0.819 position autocorrelation trap. We killed the model and shipped an evidence-backed action playbook that lifts top-tier review precision by +30pp. Full paper and open code: https://eimanzahra1472.github.io/flyrank-ml-internship-starter/"*

### 3-Sentence Employer Summary
Built an end-to-end content opportunity triage system using FlyRank's 78.8M-row search performance warehouse and a 30K-row client dataset. Demonstrated that a linear model achieves Precision@20 = 0.700 (+30pp over rule baseline) under client-grouped holdout, while identifying an 0.819 autocorrelation trap that caused a forward-looking predictive model to fail time-aware validation (0.669 -> 0.274). Deployed an interpretable, 59-page triage playbook with verifiable reason codes and explicit human-in-the-loop operational constraints.

---
### Data Credit
Built on the FlyRank ML Internship dataset — [https://flyrank.ai](https://flyrank.ai)